## Environment setup

In [ ]:
from datetime import datetime 
import importlib
from functools import partial

import tensorflow as tf
from tensorflow.data import Dataset, TFRecordDataset
import tensorflow_datasets as tfds

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib import colors
import seaborn as sns

# Notebooks run from VSCode use home directory as a base path
# while notebooks run from JupyterLab use the current directory as a base path
import sys
sys.path.append("/home/akalinow/scratch/ELITPC/TPCReco/PythonAnalysis/python/")

# Change directory to the working directory
import os
os.chdir('/scratch_hdd/akalinow/ELITPC/PythonAnalysis/')

#Increase plots font size
params = {'legend.fontsize': 'xx-large',
          'figure.figsize': (10, 7),
         'axes.labelsize': 'xx-large',
         'axes.titlesize':'xx-large',
         'xtick.labelsize':'xx-large',
         'ytick.labelsize':'xx-large'}
plt.rcParams.update(params)

### Load dataset and convert to pandas DataFrame
```python

In [ ]:
%%time 

import utility_functions as utils
importlib.reload(utils)

#dataset = tf.data.Dataset.load('MergedEvent_Track3D_TwoProng_gun_MC', compression="GZIP")
dataset = tf.data.Dataset.load('MergedEvent_Track3D_TwoProng_2022-04-12T08-03-44', compression="GZIP")

data = dataset.map(lambda x: tf.reshape(x["sim"][1], (1,-1))).unbatch().as_numpy_iterator()
df = pd.DataFrame(data, columns=utils.columnsXYZ)

data = dataset.map(lambda x: tf.reshape(x["reco"][1], (1,-1))).unbatch().as_numpy_iterator()
df_reco = pd.DataFrame(data, columns=utils.columnsXYZ)

df = df.merge(df_reco, left_index=True, right_index=True, suffixes=("_sim", "_reco"), how="outer");

### Load the model


In [ ]:
path = '/home/akalinow/scratch/ELITPC/PythonAnalysis/training/0100_2025_Jul_16_20_39_42.keras'
model = tf.keras.models.load_model(path)

In [ ]:
import plotting_functions as plf
importlib.reload(plf)

dataset_f = dataset.map(lambda x: (tf.where(x['sim'][0]>0.1, x['sim'][0], 0), x['sim'][1]))

x = iter(dataset_f.batch(1))
for i in range(5):
    item = next(x)
    plf.plotEvent(item, model=model)

In [ ]:
import plotting_functions as plf
importlib.reload(plf)

item = next(iter(dataset.batch(1)))

print(item)

#plf.plotEvent(item, model=None)

### Resolution plots in XYZ

In [ ]:
import plotting_functions as plf
importlib.reload(plf)

#plf.controlPlots(df)
plf.plotEndPointRes(df=df, edge="Vtx", coordinates=["x", "y", "z"])
plf.plotEndPointRes(df=df, edge="Alpha", coordinates=["x", "y", "z"])
plf.plotEndPointRes(df=df, edge="Carbon", coordinates=["x", "y", "z"])

plf.plotLengthPull(df, partName="Alpha")
plf.plotLengthPull(df, partName="Carbon")
plf.plotLengthPullEvolution(df)
plf.plotOpeningAngleCos(df)

### Resolution plots for filtered dataset

In [ ]:
import plotting_functions as plf
importlib.reload(plf)

mask = (df["pullAlpha"]<-10)*(np.abs(df["xAlpha_sim"])<100)*(np.abs(df["yAlpha_sim"])<90)*(np.abs(df["zAlpha_sim"])<50)

plf.controlPlots(df[mask])
plf.plotEndPointRes(df=df[mask], edge="Vtx", coordinates=["x", "y", "z"])
plf.plotEndPointRes(df=df[mask], edge="Alpha", coordinates=["x", "y", "z"])
plf.plotEndPointRes(df=df[mask], edge="Carbon", coordinates=["x", "y", "z"])

plf.plotLengthPull(df[mask], partName="Alpha")
plf.plotLengthPull(df[mask], partName="Carbon")
plf.plotLengthPullEvolution(df[mask])

np.sum(mask)


### Resolution plots in UVWT

In [ ]:
import utility_functions as utils
importlib.reload(utils)

batchSize = 32
dataset = tf.data.Dataset.load('MergedEvent_Track3D_TwoProng_gun_MC', compression="GZIP")

data = dataset.map(lambda x: tf.reshape(x["sim"][1], (1,-1)))
data = data.map(lambda x: tf.reshape(x, (-1,3,3)))
data = data.map(lambda x: utils.XYZtoUVWT_event(x))
data = data.map(lambda x: tf.reshape(x, (-1,12))).unbatch().as_numpy_iterator()

df = pd.DataFrame(data, columns=utils.columnsUVWT)


data = dataset.map(lambda x: tf.reshape(x["reco"][1], (1,-1)))
data = data.map(lambda x: tf.reshape(x, (-1,3,3)))
data = data.map(lambda x: utils.XYZtoUVWT_event(x))
data = data.map(lambda x: tf.reshape(x, (-1,12))).unbatch().as_numpy_iterator()
df_reco = pd.DataFrame(data, columns=utils.columnsUVWT)

df = df.merge(df_reco, left_index=True, right_index=True, suffixes=("_sim", "_reco"), how="outer");

In [ ]:
import plotting_functions as plf
importlib.reload(plf)

plf.plotEndPointRes(df=df, edge="Vtx", coordinates=["u", "v", "w", "t"])
plf.plotEndPointRes(df=df, edge="Alpha", coordinates=["u", "v", "w", "t"])
plf.plotEndPointRes(df=df, edge="Carbon", coordinates=["u", "v", "w", "t"])